# Robot6 코드리뷰 정리 노트북

**목적**  
코드리뷰 발표용으로 시스템 구조, 데이터 흐름, 포즈 분석 코어 동작, 주석 기준을 한 번에 정리한 노트북이다.

**중심 모듈**
- `srd_pose_emergency_core.py`
- `robot6_control_node.py`
- `rescue_nav_node_sucees.py`
- `rescue_stt_node.py`


## 1. 노드 구조

### 구성 요소별 역할
| 구성 요소 | 역할 | 주요 입력 | 주요 출력 |
|---|---|---|---|
| `rescue_nav_node_sucees.py` | 요구조자 좌표를 받아 순차 주행하고 도착 시 관제 노드로 신호를 보낸다. | `/rescue/victim_pose`, `/rescue/victim_pose_stamped` | `/robot6/mission/arrived` |
| `robot6_control_node.py` | 도착 이후 세션을 시작하고, 포즈 분석·프레이밍·집계·TTS 연동을 수행한다. | RGB/depth/camera_info, `/robot6/mission/arrived`, `/robot6/tts/done` | `/robot6/cmd_vel`, `/robot6/session/result`, `/robot6/tts/request` |
| `rescue_stt_node.py` | TTS 요청을 받아 음성 시나리오를 실행하고 종료 시 done 신호를 보낸다. | `/robot6/tts/request` | `/robot6/tts/done`, `/robot6/victim_voice_reply` |
| `srd_pose_emergency_core.py` | ROS2 노드가 아닌 순수 분석 엔진이다. | OpenCV BGR frame | annotated frame, result list |

### 연결 흐름
1. 내비게이션 노드가 요구조자 좌표를 받아 이동한다.
2. 목표 지점 도착 후 `/robot6/mission/arrived` 를 발행한다.
3. 관제 노드가 SEARCH → FRAME → VERIFY → MEASURING 상태로 세션을 진행한다.
4. 세션 종료 시 결과 JSON을 발행하고 TTS 요청을 보낸다.
5. 대화 시나리오가 끝나면 `/robot6/tts/done` 을 받아 세션을 마무리한다.


## 2. 코드 동작 설명

### 2-1. 토픽·액션·데이터 형식
| 구분 | 이름 | 데이터 형식 | 방향 | 설명 |
|---|---|---|---|---|
| Topic | `/rescue/victim_pose` | `geometry_msgs/PointStamped` | 입력(nav) | 요구조자 위치를 점 좌표로 수신 |
| Topic | `/rescue/victim_pose_stamped` | `geometry_msgs/PoseStamped` | 입력(nav) | 요구조자 위치를 자세 포함 좌표로 수신 |
| Topic | `/robot6/mission/arrived` | `std_msgs/Bool` | 출력(nav) / 입력(control) | 목표 지점 도착 여부 |
| Topic | `/robot6/oakd/rgb/image_raw/compressed` | `sensor_msgs/CompressedImage` | 입력(control) | JPEG 압축 RGB 영상 |
| Topic | `/robot6/oakd/stereo/image_raw/compressedDepth` | `sensor_msgs/CompressedImage` | 입력(control) | compressedDepth 형식 깊이 영상 |
| Topic | `/robot6/oakd/stereo/camera_info` | `sensor_msgs/CameraInfo` | 입력(control) | 카메라 내부 파라미터 K |
| Topic | `/robot6/cmd_vel` | `geometry_msgs/Twist` | 출력(control) | yaw 보정 및 후진 제어 |
| Topic | `/robot6/session/result` | `std_msgs/String(JSON)` | 출력(control) | 세션 종료 후 최종 결과 |
| Topic | `/robot6/session/status` | `std_msgs/String(JSON)` | 출력(control) | 현재 상태와 세션 플래그 |
| Topic | `/robot6/tts/request` | `std_msgs/String(JSON)` | 출력(control) / 입력(stt) | TTS 요청 텍스트와 세션 결과 |
| Topic | `/robot6/tts/done` | `std_msgs/Bool` | 출력(stt) / 입력(control) | 음성 시나리오 종료 보고 |
| Topic | `/robot6/srd/image_result/compressed` | `sensor_msgs/CompressedImage` | 출력(control) | annotated 영상 |
| Action | `/robot6/dock` | `irobot_create_msgs/action/Dock` | 출력(nav) | 도킹 요청 |
| Action | `/robot6/undock` | `irobot_create_msgs/action/Undock` | 출력(nav) | 언도킹 요청 |
| Service | 현재 없음 | - | - | topic과 action 중심 구조 |

### 2-2. 이미지 데이터 설명
- RGB 영상은 `sensor_msgs/CompressedImage` 이며 실제 내용은 JPEG 압축 바이트이다.
- 관제 노드는 콜백에서 raw bytes만 저장하고 `step()`에서 지금 처리할 1장만 디코딩한다.
- depth 영상도 `compressedDepth` 바이트로 저장한 뒤, 필요할 때만 디코딩한다.
- 입력 카메라 FPS와 분석 FPS는 분리되어 있다. 현재 분석 주기는 기본 10Hz 기준이다.
- 픽셀(px)은 카메라 드라이버 해상도에 따르며, 코드에 고정 해상도 상수는 없다.


## 3. 포즈 분석 코어 동작 흐름

### 3-1. 핵심 진입점
- `analyze_frame_with_results(frame)`
- `analyze_frame(frame)`
- `results_to_json(results)`
- `analyze_frame_with_emergency_level(frame)`

### 3-2. 프레임 1장 처리 순서
1. YOLO Pose 모델로 박스와 keypoint를 추론한다.
2. 추적 ID(track_id)를 확보한다.
3. 각 사람에 대해 `observation` 을 판정한다.
4. `posture` 를 판정한다.
5. 이전 프레임과 비교해 `motion` 을 계산한다.
6. `trapped` 가능성을 계산한다.
7. `seen_sec`, `state_sec` 을 갱신한다.
8. `emergency_level` 을 결정한다.
9. annotated frame 과 result dict 를 반환한다.

### 3-3. result dict 필드
- `track_id`
- `bbox`
- `observation`
- `posture`
- `motion`
- `emergency_level`
- `trapped`
- `seen_sec`
- `state_sec`
- `shoulder_tilt`
- `head_drop_ratio`
- `torso_angle`
- `motion_smooth`
- `motion_upper`
- `motion_core`


## 4. 관제 노드 상태머신

- `WAIT_ARRIVAL`
- `SEARCH_PERSON`
- `FRAME_PERSON`
- `VERIFY_FRAME`
- `MEASURING`
- `RESULT_LOCKED`
- `WAIT_TTS_DONE`
- `SESSION_END`

### 세션 결과 구조 예시
```json
{
  "overall": {"emergency_peak": "WARNING", "emergency_majority": "CAUTION"},
  "body_summary": {"full_body": {}, "upper_body": {}, "partial": {}},
  "victim_position": {"frame_id": "map", "x": -2.184, "y": -0.947},
  "quality": {"stable_frame_count": 42, "valid_position_samples": 18}
}
```


## 5. 코드 주석 기준

### 권장 방식
- 파일/모듈 설명: `"""docstring"""`
- 함수 설명: `"""docstring"""`
- 한 줄 설명: `#`
- 블록 설명: `#` 연속 주석

### 피해야 할 방식
- `'이건 주석처럼 보이지만 실제로는 문자열 리터럴'`

### 정리 원칙
1. 주석은 함수 이름을 반복하지 말고 이유와 단계만 설명한다.
2. 상태머신은 전이 이유를 함께 적는다.
3. 계산식은 입력과 출력 의미를 먼저 적는다.
4. 발표용 설명 주석과 디버그 임시 메모는 분리한다.


## 6. 코드리뷰 발표 구성안

1. 전체 노드 구조와 연결 흐름 설명
2. 포즈 코어가 ROS 노드가 아니라 분석 엔진이라는 점 강조
3. `analyze_frame_with_results()` 내부 파이프라인 설명
4. result dict 와 `emergency_level` 결정 로직 설명
5. 관제 노드가 이 결과를 어떻게 세션화하는지 설명

### 바로 답할 수 있어야 하는 질문
- 왜 core를 node가 아니라 library로 분리했는가?
- `FULL_BODY / UPPER_BODY / PARTIAL` 을 나누는 이유는 무엇인가?
- 단발 프레임이 아니라 `state_sec` 을 왜 사용하는가?
- `robot6_control_node` 와 `srd_pose_emergency_node` 의 차이는 무엇인가?
- `victim_position` 은 어떤 좌표계(map) 기준으로 보내는가?
